# Binance Depth Stream Exploration

**Project:** MarketStream — Phase 1, Day 2  
**Data source:** Binance US WebSocket depth stream, topic `btcusdt_depth` on Kafka  

## What this notebook is for

Before building an order book reconstructor you need to understand the raw data.
This notebook fetches a live REST snapshot, consumes real Kafka messages, and inspects
the sequence properties that make correct reconstruction possible.

## The synchronization problem

The Binance depth stream sends **incremental diffs**, not full book state.
Each message updates a subset of price levels on one or both sides of the book.
To build a correct local order book you must:

1. **Bootstrap** from a REST snapshot (`/api/v3/depth`) which gives you a consistent
   starting state tagged with a `lastUpdateId`.
2. **Anchor** the stream to that snapshot by discarding buffered messages where
   `u <= lastUpdateId`, then finding the first message where `U <= lastUpdateId + 1`.
3. **Validate continuity** on every subsequent message: `current.U` must equal
   `previous.u + 1`. A gap means you missed at least one update and your book
   is now corrupt. The only recovery is a full resync.

A quantity of `"0.00000000"` is not a resting order — it is a deletion instruction.
That price level must be removed from the book entirely.

---

In [1]:
# Install notebook dependencies (safe to re-run)
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "ipykernel"])


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import json
import ssl
import urllib.request
from datetime import datetime, timezone

import certifi
import pandas as pd
from kafka import KafkaConsumer

---
## Section 1 — REST Snapshot

The REST snapshot is the only way to get a consistent starting state for the book.
The `lastUpdateId` it returns is the sequence number you use to anchor the stream.

In [3]:
SNAPSHOT_URL = "https://api.binance.us/api/v3/depth?symbol=BTCUSDT&limit=20"

ssl_context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(SNAPSHOT_URL, context=ssl_context) as resp:
    snapshot = json.loads(resp.read())

last_update_id = snapshot["lastUpdateId"]
print(f"lastUpdateId: {last_update_id}")
print(f"Total bid levels returned: {len(snapshot['bids'])}")
print(f"Total ask levels returned: {len(snapshot['asks'])}")

lastUpdateId: 3163449098
Total bid levels returned: 20
Total ask levels returned: 20


In [4]:
def build_depth_table(levels: list[list[str]], descending: bool = False) -> pd.DataFrame:
    """Build a price/quantity/cumulative table from a list of [price, qty] pairs."""
    sorted_levels = sorted(levels, key=lambda x: float(x[0]), reverse=descending)
    rows = []
    cumulative = 0.0
    for price_str, qty_str in sorted_levels:
        qty = float(qty_str)
        cumulative += qty
        rows.append({
            "Price (USDT)": f"{float(price_str):,.2f}",
            "Quantity (BTC)": f"{qty:.8f}",
            "Cumulative Qty (BTC)": f"{cumulative:.8f}",
        })
    return pd.DataFrame(rows)


top5_bids = snapshot["bids"][:5]  # REST returns bids best-first (descending)
top5_asks = snapshot["asks"][:5]  # REST returns asks best-first (ascending)

print("=== Top 5 Bids (best bid first) ===")
display(build_depth_table(top5_bids, descending=True))

print("\n=== Top 5 Asks (best ask first) ===")
display(build_depth_table(top5_asks, descending=False))

=== Top 5 Bids (best bid first) ===


,Price (USDT),Quantity (BTC),Cumulative Qty (BTC)
0,"67,285.63",0.03236000,0.03236000
1,"67,272.69",0.00156000,0.03392000
2,"67,270.35",0.03721000,0.07113000
3,"67,269.58",0.20779000,0.27892000
4,"67,260.29",0.04563000,0.32455000



=== Top 5 Asks (best ask first) ===


,Price (USDT),Quantity (BTC),Cumulative Qty (BTC)
0,"67,297.44",0.03720000,0.03720000
1,"67,305.84",0.03304000,0.07024000
2,"67,321.10",0.04408000,0.11432000
3,"67,330.98",0.12497000,0.23929000
4,"67,334.99",0.56835000,0.80764000


### Section 1 Observations

- **The spread is the gap between the best ask and best bid.** These two prices are the
  most important rows in the entire book — every trade either lifts the ask or hits the bid.
  The spread in basis points reflects the current liquidity cost for a market order.

- **Cumulative quantity grows unevenly.** Some price levels have much larger resting size
  than adjacent levels — these are limit order clusters, often placed by market makers
  at round numbers or at technical levels. They act as temporary support/resistance.

- **The snapshot is already stale the moment it arrives.** The `lastUpdateId` will be
  several hundred IDs behind the live stream by the time we read it. This is expected —
  it is why we buffer stream messages *before* fetching the snapshot, not after.

---
## Section 2 — Consume 10 Kafka Messages

Each message is an incremental diff. We inspect the structure of real events:
how many levels change per message, how many are deletions, and what the update ID spans look like.

In [5]:
consumer = KafkaConsumer(
    "btcusdt_depth",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="latest",
    enable_auto_commit=False,
    consumer_timeout_ms=10_000,
    value_deserializer=lambda raw: json.loads(raw.decode("utf-8")),
)

messages = []
for record in consumer:
    messages.append(record.value)
    if len(messages) >= 10:
        break

consumer.close()
print(f"Collected {len(messages)} messages.")

Collected 10 messages.


In [6]:
def fmt_event_time(millis: int) -> str:
    return datetime.fromtimestamp(millis / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " UTC"


for i, msg in enumerate(messages, start=1):
    span = msg["u"] - msg["U"] + 1

    bid_changes = msg["b"]
    ask_changes = msg["a"]
    bid_deletions = sum(1 for _, q in bid_changes if q == "0.00000000")
    ask_deletions = sum(1 for _, q in ask_changes if q == "0.00000000")

    print(f"Message {i:2d}:")
    print(f"  Event time   : {fmt_event_time(msg['E'])}")
    print(f"  Update range : U={msg['U']}  →  u={msg['u']}  ({span} ID{'s' if span != 1 else ''} batched)")
    print(f"  Bid changes  : {len(bid_changes):3d} total   {bid_deletions:3d} deletions   {len(bid_changes) - bid_deletions:3d} inserts/updates")
    print(f"  Ask changes  : {len(ask_changes):3d} total   {ask_deletions:3d} deletions   {len(ask_changes) - ask_deletions:3d} inserts/updates")
    print()

Message  1:
  Event time   : 2026-06-02 18:32:55.270 UTC
  Update range : U=3163449058  →  u=3163449131  (74 IDs batched)
  Bid changes  :  25 total    22 deletions     3 inserts/updates
  Ask changes  :  15 total    10 deletions     5 inserts/updates

Message  2:
  Event time   : 2026-06-02 18:32:56.270 UTC
  Update range : U=3163449132  →  u=3163449177  (46 IDs batched)
  Bid changes  :  16 total    12 deletions     4 inserts/updates
  Ask changes  :  12 total     9 deletions     3 inserts/updates

Message  3:
  Event time   : 2026-06-02 18:32:57.270 UTC
  Update range : U=3163449178  →  u=3163449320  (143 IDs batched)
  Bid changes  :  50 total    42 deletions     8 inserts/updates
  Ask changes  :  30 total    23 deletions     7 inserts/updates

Message  4:
  Event time   : 2026-06-02 18:32:58.270 UTC
  Update range : U=3163449321  →  u=3163449447  (127 IDs batched)
  Bid changes  :  42 total    39 deletions     3 inserts/updates
  Ask changes  :  16 total    12 deletions     4 ins

### Section 2 Observations

- **Deletions often outnumber inserts.** In quiet periods most book activity is order
  cancellations — market makers pulling quotes — rather than new orders being placed.
  A high deletion ratio is normal and does not indicate abnormal market conditions.

- **Some messages batch multiple update IDs (span > 1).** This happens when several
  book-altering events occur between WebSocket flush intervals. The message is still
  one atomic diff — you apply it as a unit, not individual sub-updates.

- **Messages with empty bid or ask arrays are valid.** A one-sided message means only
  that side of the book changed in this flush window. Your reconstructor must handle
  `"b": []` and `"a": []` without error — they are common, not edge cases.

---
## Section 3 — Update ID Continuity Check

The invariant that must hold for every consecutive message pair is:

```
current.U == previous.u + 1
```

A single failure means at least one diff was lost. The local order book is corrupt
and must be rebuilt from a fresh snapshot — there is no partial recovery.

In [7]:
if len(messages) < 2:
    print("Need at least 2 messages — re-run Section 2 to collect more.")
else:
    all_pass = True
    rows = []

    for i in range(1, len(messages)):
        prev = messages[i - 1]
        curr = messages[i]
        expected_U = prev["u"] + 1
        actual_U   = curr["U"]
        passed     = actual_U == expected_U

        if not passed:
            all_pass = False

        rows.append({
            "Transition":  f"msg {i} → msg {i + 1}",
            "prev.u":       prev["u"],
            "curr.U":       actual_U,
            "Expected U":   expected_U,
            "Gap (IDs)": actual_U - expected_U,
            "Result":       "✅ PASS" if passed else "❌ FAIL",
        })

    df = pd.DataFrame(rows)
    display(df)

    print()
    if all_pass:
        print("All continuity checks PASSED — no sequence gaps in this sample.")
        print("This sample is safe to use as a bootstrap test for the reconstructor.")
    else:
        gap_count = df[df["Result"].str.contains("FAIL")].shape[0]
        print(f"{gap_count} gap(s) detected. Order book resync would be mandatory at each FAIL point.")
        print("Do not use this sample to test the reconstructor without handling the gaps.")

,Transition,prev.u,curr.U,Expected U,Gap (IDs),Result
0,msg 1 → msg 2,3163449131,3163449132,3163449132,0,✅ PASS
1,msg 2 → msg 3,3163449177,3163449178,3163449178,0,✅ PASS
2,msg 3 → msg 4,3163449320,3163449321,3163449321,0,✅ PASS
3,msg 4 → msg 5,3163449447,3163449448,3163449448,0,✅ PASS
4,msg 5 → msg 6,3163449544,3163449545,3163449545,0,✅ PASS
5,msg 6 → msg 7,3163449608,3163449609,3163449609,0,✅ PASS
6,msg 7 → msg 8,3163449710,3163449711,3163449711,0,✅ PASS
7,msg 8 → msg 9,3163449786,3163449787,3163449787,0,✅ PASS
8,msg 9 → msg 10,3163449830,3163449831,3163449831,0,✅ PASS



All continuity checks PASSED — no sequence gaps in this sample.
This sample is safe to use as a bootstrap test for the reconstructor.


### Section 3 Observations

- **A PASS on 9 transitions from 10 messages is expected in normal operation** but not
  guaranteed. Consumer lag, network hiccups, or a Kafka broker restart can all introduce
  gaps. The reconstructor must treat a gap as a hard error, not a soft warning.

- **The `Gap (IDs)` column quantifies data loss.** A gap of 1 means one individual diff
  was lost. A gap of 50 means 50 individual book updates are missing. Either way the
  recovery procedure is identical: discard the current book and resync from REST.

- **This check only catches detected gaps.** If messages arrive out of order and the
  consumer happens to read them in the wrong sequence, the check could pass while the
  book is still being assembled incorrectly. A production reconstructor also needs a
  wall-clock freshness check on `E` to detect stalled streams.